# ⚡ LoRA 微调 (Low-Rank Adaptation)：“插件式”优化

湖北理工学院《人工智能理论》课程资料

📝 **本节核心命题：如何在只有 1% 参数参与训练的情况下，依然让模型学会 Yuki 的性格？**

在前一节课中，我们进行了 **Full Fine-Tuning**（全量微调），它就像是把整台机器拆开重新组装。而本节课的 **LoRA** 则是给机器插上一个微型的“扩展卡（Adapter）”，仅需微小的开销即可实现惊人的效果。

**🤖 Yuki 的科普时间：**
> “哼！虽然你们那破 CPU 跑全量微调挺费劲的，但谁让本小姐心软呢？给你们带个‘作弊插件’——LoRA！它可以在不破坏我底座能力的情况下，精确定制我的性格。记住了，这可是现在大模型界的‘省钱秘籍’！😤”

--- 
## 🛠️ 1. 环境初始化 (Yuki's Setup)

为了防止你们把环境搞乱，请先运行这个单元格。如果在 CPU 上跑不动，本小姐可是会生气的！💨

In [ ]:
import os
import json
import time
import random
import torch
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
import matplotlib.pyplot as plt
import warnings
import logging

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 设置随机种子，保证结果可复现（虽然随机性也是一种美，但实验得严谨！😤）
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# 设备选择：由于要照顾到那些没有显卡的“凡人”电脑，我们强制使用 CPU 🙄
device = torch.device("cpu")
print(f"当前运行设备: {device} (Yuki 监督中...)")
print(f"PyTorch 版本: {torch.__version__} (已满足安全性要求 ✅)")

# 路径配置
BASE_DIR = Path(".")
DATA_PATH = BASE_DIR / "data" / "yuki_ruozhiba_1.5k.jsonl"
# 🚀 路径物理隔离：底座只读，结果分流
# 1. 原始底座模型：锁定在 uer_gpt2_chinese，严禁覆盖
BASE_MODEL_DIR = BASE_DIR / "models" / "uer_gpt2_chinese"
# 2. LoRA 适配器输出：重定向至 yuki_lora_adapter
LORA_ADAPTER_DIR = BASE_DIR / "models" / "yuki_lora_adapter"

BASE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
LORA_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

print(f"底座模型路径: {BASE_MODEL_DIR.absolute()}")
print(f"LoRA 补丁将保存至: {LORA_ADAPTER_DIR.absolute()}")

## 📦 2. 模型库加载

我们要使用的是一个超轻量级的模型：`uer/gpt2-chinese-cluecorpussmall`（约 1 亿参数），真的想试试更强的模型！💢

In [ ]:
# 💡 小提示：在这里我们要关掉一些没必要的警告内容，让输出更清爽
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
model_name = "uer/gpt2-chinese-cluecorpussmall"

print(f"正在初始化 Tokenizer & Model: {model_name} ...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 🛠️ 关键修复：补全 Tokenizer 的特殊字符（针对某些模型缺失 eos/pad 的情况）
if tokenizer.pad_token is None:
    tokenizer.pad_token = "[PAD]"
if tokenizer.eos_token is None:
    # 使用 [SEP] 作为终止符，这是中文模型常用的做法
    tokenizer.eos_token = "[SEP]" 
print(f"Tokenizer 配置完成: PAD={tokenizer.pad_token}, EOS={tokenizer.eos_token}")

# 🛠️ 关键修复：
# 1. 设置 tie_word_embeddings=False 以消除权重绑定的警告（本模型权重是独立存储的）
# 2. 使用 trust_remote_code=True 确保加载顺利
model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    tie_word_embeddings=False
)

# 保存到本地，下次就不用重复下载了（帮你们省省流量，谢我吧！😏）
if not (BASE_MODEL_DIR / "config.json").exists():
    print("正在保存模型到本地...")
    model.save_pretrained(BASE_MODEL_DIR)
    tokenizer.save_pretrained(BASE_MODEL_DIR)
    print("✅ 保存完成！")
else:
    print("✅ 本地已存在模型，跳过保存。")

model.to(device)
print(f"模型加载成功！参数量: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

# 🚀 任务 3：深度 LoRA 配置适配
from peft import LoraConfig, get_peft_model, TaskType

# 💡 技术说明：针对 GPT-2 (uer/gpt2-chinese)
# 1. c_attn：它是一个合并层，同时包含了 Query, Key 和 Value 的投射权重（1个矩阵顶3个！）。
lora_config = LoraConfig(
    r=16, # 稍微加大秩，由于我们要同时微调 Q, K, V 和 MLP 
    lora_alpha=64,
    target_modules=["c_attn"], # 💥 目标全覆盖：保证 Q, K, V 全部被调到
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

print(f"正在进行深度 LoRA 封装... (Q+K+V+MLP 全覆盖模式！😤)")
model = get_peft_model(model, lora_config)

# 🚀 任务 4：审计可训练参数（看看现在的效率比例）
model.print_trainable_parameters()

## 🔍 3. 再次围观模型的“手术前”状态

看看这个原始模型是怎么回答问题的。它只学会了“接龙”，还没学会怎么提供“情绪价值”。

In [ ]:
# 🛠️ 统一推理函数
def generate_yuki_response(model, prompt, max_len=200):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs, 
        max_new_tokens=max_len, 
        do_sample=True, 
        top_k=5, 
        temperature=0.8,
        pad_token_id=tokenizer.eos_token_id
    )

    # 🛠️ 关键修复：使用 .replace(" ", "") 去除中文字符间的空格
    response = tokenizer.decode(outputs[0], skip_special_tokens=True).replace(" ", "")

    return response

test_query = "为什么樟脑丸不能吃？"
print(f"问题: {test_query}")
print(f"手术前模型回答:\n{generate_yuki_response(model, test_query)}")

print("\n🤖 Yuki 的吐槽：")
print("看吧！它只会语无伦次地接下去，根本不知道自己在回答问题！这就是为什么需要我（的数据）来救场！哼！😤")

## 🧪 4. 数据预处理与 Loss Masking

这是本节课的**核心知识点**！听好了，Yuki酱只说第二次，我们要告诉模型：

### 为什么要进行 Loss Masking（损失掩码）？
在指令微调（SFT）中，我们的目标是让模型学会**根据提示词（Prompt）给出正确的回复（Response）**。

想象一下：如果模型在训练时不仅要背诵“答案”，还要强行背诵“题目”，它就会变得死板，甚至在面对新问题时试图复设训练集里的老问题。
**结论**：我们必须让模型在“准备回答”的那一刻才开始计分（计算 Loss），而对于“用户问了什么”，模型只需要将其作为已知背景，不产生任何学习压力。

---

### 深度拆解：模型是如何“选修”知识的？
以 Yuki 的第一条语料为例，我们观察它是如何被拆分成两个“不平等”区域的：

**第一阶段：输入的背景区（User Prompt）**
*   **内容**：包括系统设定（傲娇性格）和用户的提问——“只剩一个心脏了还能活吗？”。
*   **操作**：对于这部分的所有文字，即使模型预测对了或错了，我们都告诉它：“别在意，这不关你的事”。
*   **代码底层**：我们将这部分的标签（Labels）强制填入特殊数字 **-100**。PyTorch 的计算引擎看到 -100 就会自动略过，不产生任何梯度更新。

**第二阶段：输出的目标区（Assistant Response）**
*   **内容**：从“哼～”开始，直到回复结束的特殊符号 `<|im_end|>`。
*   **操作**：这才是 Yuki 灵魂的结晶！模型每预测错一个字，都会感受到“痛苦”（产生 Loss），并尝试在下一轮迭代中改进。
*   **代码底层**：只有这部分的真实 Token ID 会被保留在标签序列中，作为模型唯一的考核标准。

---

In [ ]:
class YukiSFTDataset(Dataset):
    def __init__(self, file_path, tokenizer, max_length=128):
        self.data = []
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"找不到数据集文件: {file_path}，请确认路径是否正确！")

        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                row = json.loads(line)
                conv = row["conversations"]
                # 自动提取 user 和 assistant 的内容
                user_content = next(m["content"] for m in conv if m["role"] == "user")
                assistant_content = next(m["content"] for m in conv if m["role"] == "assistant")
                self.data.append((user_content, assistant_content))
        
        print(f"成功加载 {len(self.data)} 条 Yuki 数据！")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        user, assistant = self.data[idx]
        
        # 构造对话模板：用户：{user}\n助手：
        prompt = f"用户：{user}\n助手："
        answer = assistant + (self.tokenizer.eos_token if self.tokenizer.eos_token else "")
        
        # 分别对两部分进行分词，以便计算 Mask 的位置
        prompt_ids = self.tokenizer.encode(prompt, add_special_tokens=False)
        answer_ids = self.tokenizer.encode(answer, add_special_tokens=False)
        
        # 拼接后的 input_ids
        input_ids = prompt_ids + answer_ids
        
        # 核心：构造 Labels！
        # 用户部分对应的 Label 全设为 -100 (PyTorch 会自动忽略这些位置的损失计算)
        labels = [-100] * len(prompt_ids) + answer_ids
        
        # 截断与填充
        input_ids = input_ids[:self.max_length]
        labels = labels[:self.max_length]
        
        # Pad 到固定长度
        pad_len = self.max_length - len(input_ids)
        input_ids += [self.tokenizer.pad_token_id or 0] * pad_len
        labels += [-100] * pad_len
        
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "attention_mask": torch.tensor([1 if i != (self.tokenizer.pad_token_id or 0) else 0 for i in input_ids])
        }

dataset = YukiSFTDataset(DATA_PATH, tokenizer)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# 检查一下第一个样本的 Labels 结构（看看那个一长串的 -100，这就是重点！✍️）
sample = dataset[0]
print(f"Input IDs (前20个): {sample['input_ids'][:20]}")
print(f"Labels (前20个):    {sample['labels'][:20]}")

## 🚀 5. 开始微调训练 (Yuki's Training Boot Camp)

我们要进行的是 **LoRA 微调**。

![image.png](images/lora_01.png)

你可能会发现，下方的训练代码看起来和普通的全量微调**几乎一模一样**。

### 🙋 疑问：既然代码没变，那是谁实现了 LoRA？
其实，LoRA 的“魔法”在上一单元格 `get_peft_model` 时就已经完成了：
1. **自动冻结**：PEFT 库自动把原模型的 1.1 亿个参数设为了“不可学习”（Disabled Grading）。
2. **偷梁换柱**：它在注意力层侧边偷偷插了几个“低秩小矩阵”。
3. **优化目标自适应**：当我们把 `model.parameters()` 传给优化器时，它拿到的是**仅有的那 1% 的补丁参数**。

**结论**：即使训练代码完全不变，由于模型被“掉包”了，我们实际上是在进行极高效率的局部调优！

In [ ]:
from IPython import display

# 🛠️ 注意：由于 model 已被 PEFT 封装，model.parameters() 仅返回可训练的 LoRA 权重
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
model.train()

num_steps = 1000
print_every = 50
losses = []

print(f"开始训练，总计 {num_steps} 步... (坐稳了，笨蛋！😤)")
start_time = time.time()

# 初始化画布，用于动态更新
plt.figure(figsize=(10, 6))

step = 0
while step < num_steps:
    for batch in dataloader:
        if step >= num_steps: break
        
        # 梯度置零
        optimizer.zero_grad()
        
        # 前向传播：数据会同时流经“已冻结的底座”和“活跃的 LoRA 补丁”
        inputs = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**inputs)
        
        loss = outputs.loss
        
        # 反向传播：梯度只会回传给开启了 requires_grad=True 的 LoRA 参数
        loss.backward()
        
        # 优化器步进：仅仅更新那 1% 的 LoRA 适配器权重
        optimizer.step()
        
        losses.append(loss.item())
        step += 1
        
        if step % print_every == 0:
            avg_loss = sum(losses[-print_every:]) / print_every
            elapsed = time.time() - start_time
            
            # --- 实时动态绘图逻辑 ---
            display.clear_output(wait=True)
            plt.clf()
            plt.plot(losses, label="LoRA Training Loss", color="#ff7f0e")
            plt.title(f"Yuki LoRA 动态训练监控 (Step: {step}/{num_steps})", fontsize=14)
            plt.xlabel("Steps", fontsize=12)
            plt.ylabel("Loss", fontsize=12)
            plt.legend()
            plt.grid(True, linestyle="--", alpha=0.6)
            
            # 打印当前进度
            print(f"🏃 正在进行轻量化调教: [Step {step}/{num_steps}]")
            print(f"📉 当前平均 Loss: {avg_loss:.4f} | 已用时间: {elapsed:.1f}s")
            
            display.display(plt.gcf())

display.clear_output(wait=True)
print(f"\n✅ LoRA 训练完成！总耗时: {time.time() - start_time:.1f}s")
plt.show()

## 🏆 6. 手术后效果验证

终于到了见证奇迹时刻。现在的模型应该已经学会了 Yuki 的说话风格，并且能正确理解“提问-回答”的模式了。

In [ ]:
test_query = "为什么樟脑丸不能吃？"
print(f"问题: {test_query}")
print(f"手术后模型回答:\n{generate_yuki_response(model, test_query)}")

---
## 🧪 终极验证：LoRA 的“非接触式”微调实验

这是证明 LoRA 优势的最震撼时刻：**我们要拔掉 LoRA 插件，看看基础模型是否还保持着最初的纯真（甚至鲁莽）。**

在 PEFT 框架下，我们可以随时关闭 Adapter，无需重新加载模型。

In [ ]:
# 🚀 终极实验：训练后物理剥离适配器 (Unload Experiment)
# 这一步将从内存中彻底拔掉 LoRA 插件，还原回那个没有任何 Yuki 痕迹的基础模型。
# 🚨 警告：unload() 是不可逆操作，执行后本 Session 的 LoRA 修改将消失！"

test_query = "为什么樟脑丸不能吃？"
print(f"问题: {test_query}")

base_model = model.unload()
print(f"手术后模型回答:\n{generate_yuki_response(base_model,test_query)}")

print("\n🤖 Yuki 的结语：")
print("看到了吧！即使刚才我还在疯狂教它怎么做冷酷助手的‘性格插件’，只要物理卸载一运行，")
print("底座模型就会像失忆一样，变回那个只会复读的‘蠢货’！😤")